# 2026-11-19 allele-evoepi full-grid p2p trait log

Animates strains of each PE's sampled host (0th population member) over time, from the trait logger of wse-async-ga workflow `2026-11-19-alleleevoepi-fullgrid-p2p-traitlog` (genome `genome_alleleevoepi_2026_11_19`, 2 sites).

The trait logger multiplexes three single-bit records per PE (infected, site 0 allele, site 1 allele); downstream demultiplexes them by the `dstream_storage_multiplex` column.

Colors: black, uninfected; white, wildtype; blue, site 0 mutant; red, site 1 mutant; purple, double mutant.

In [ ]:
%load_ext watermark

In [ ]:
import os

import downstream
import imageio
from IPython.display import display
import numpy as np
import polars as pl
from tqdm import tqdm

import pylib  # noqa: F401
from pylib.munge._addend_groups import addend_groups
from pylib.munge._squeeze import squeeze_int
from pylib.munge._union_upsample import union_upsample

In [ ]:
%watermark -diwmuv -iv

In [ ]:
outdir = "2026-11-19-alleleevoepi-fullgrid-p2p-traitlog"
os.makedirs(outdir, exist_ok=True)
outdir

## Get Data

In [ ]:
# trait logger records, one OSF file per run
osf_ids = ["hu2w5", "cxkqu", "bezac", "dtc5w", "v5hjw"]

In [ ]:
config_columns = [
    "dispatchSlug",
    "globalSeed",
    "nRow",
    "nCol",
    "nRowSubgrid",
    "nColSubgrid",
    "tilePopSize",
    "tournSize",
    "nCycle",
    "evoepi_waning_rate",
    "evoepi_seed_pe_stride",
    "ASYNC_GA_P2P_SEND_PROBABILITY",
    "dstream_storage_multiplex",
]

## Unpack Data

In [ ]:
def unpack_strain_timeseries(df: pl.DataFrame) -> pl.DataFrame:

    # sample every 64th tile
    df = df.filter((pl.col("col") % 8 == 0) & (pl.col("row") % 8 == 0))

    # unpack dstream data items from logger hex strings, demultiplexing
    # infected, site 0 allele, and site 1 allele records
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select("dstream_data_id", "col", "row", "tile").unique(
            "dstream_data_id",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_Tbar, then one column per record
    df = (
        df.with_columns(
            dstream_Tbar=pl.col("dstream_Tbar") * pl.col("dstream_T_dilation"),
        )
        .pivot(
            on="dstream_storage_multiplex_id",
            index=["tile", "row", "col", "dstream_Tbar"],
            values="dstream_value",
        )
        .rename({"0": "infected", "1": "site0", "2": "site1"})
    )

    # addend time series groups with final uninfected entry, after logger's
    # last seen time (i.e., last time any record was set)
    df = addend_groups(
        df,
        group_by="tile",
        aggs=(
            pl.col("col").unique().pipe(squeeze_int),
            pl.col("row").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("infected"),
            pl.lit(0).alias("site0"),
            pl.lit(0).alias("site1"),
        ),
        inner_only=True,
    )

    # upsample to fill in dstream_Tbar dropped by some loggers
    df = union_upsample(
        df,
        upsample="dstream_Tbar",
        group_by="tile",
        fill_null_ops=[
            (pl.all(), {"strategy": "forward"}),
            (pl.all(), {"strategy": "backward"}),
        ],
    )
    assert len(df["dstream_Tbar"].value_counts()["count"].unique()) == 1

    # 0: uninfected, 1: wildtype, 2: site 0 mutant, 3: site 1 mutant,
    # 4: double mutant
    return df.with_columns(
        strain=pl.col("infected") * (1 + pl.col("site0") + 2 * pl.col("site1")),
    )

In [ ]:
# read and unpack one run at a time, keeping only unpacked strains
configs, dfs = [], {}
for osf_id in tqdm(osf_ids):
    df = pl.read_parquet(
        f"https://osf.io/{osf_id}/download",
    ).with_row_index("dstream_data_id")
    configs.append(
        df.select(config_columns).unique().with_columns(osf_id=pl.lit(osf_id)),
    )
    dfs[osf_id] = unpack_strain_timeseries(df)
    del df

configs = pl.concat(configs)
assert len(configs) == len(osf_ids)
display(configs)

In [ ]:
display(
    pl.concat(
        [
            df.group_by("strain").len().with_columns(osf_id=pl.lit(osf_id))
            for osf_id, df in dfs.items()
        ],
    ).pivot(on="strain", index="osf_id", values="len", sort_columns=True),
)

## Animate Data

In [ ]:
palette = np.array(
    [
        [0, 0, 0],  # uninfected: black
        [255, 255, 255],  # wildtype: white
        [0, 0, 255],  # site 0 mutant: blue
        [255, 0, 0],  # site 1 mutant: red
        [160, 32, 240],  # double mutant: purple
    ],
    dtype=np.uint8,
)

In [ ]:
for osf_id, df in tqdm(dfs.items()):
    config = configs.filter(pl.col("osf_id") == osf_id).row(0, named=True)
    label = (
        f"p2pmode={config['dispatchSlug'].removeprefix('p2p-')}"
        f"+subgrid={config['nRowSubgrid']}"
        f"+waning={config['evoepi_waning_rate']:.2f}"
        f"+seed={config['globalSeed']}"
    )
    print(osf_id, label)

    df = df.with_columns(
        row_rank=pl.col("row").rank(method="dense").cast(pl.Int32) - 1,
        col_rank=pl.col("col").rank(method="dense").cast(pl.Int32) - 1,
    )
    num_rows = df["row_rank"].max() + 1
    num_cols = df["col_rank"].max() + 1

    frames = []
    for (dstream_Tbar,), dff in df.sort("dstream_Tbar").group_by(
        "dstream_Tbar", maintain_order=True
    ):
        frame = np.zeros((num_rows, num_cols), dtype=np.uint8)
        frame[dff["row_rank"], dff["col_rank"]] = dff["strain"]
        frame = np.kron(palette[frame], np.ones((4, 4, 1), dtype=np.uint8))
        frames.append(frame)

    print(len(frames))
    outpath = f"{outdir}/a=strains+{label}+osf={osf_id}+ext="
    imageio.mimsave(f"{outpath}.gif", frames, duration=0.05, loop=0)
    imageio.mimsave(f"{outpath}.mp4", frames, fps=30, macro_block_size=1)